# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import os
if not os.path.exists("flyrank-ml"):
    !git clone https://github.com/Hashir9099/flyrank-ml.git
os.chdir("flyrank-ml")
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

Cloning into 'flyrank-ml'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (196/196), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 196 (delta 84), reused 100 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (196/196), 1.90 MiB | 5.09 MiB/s, done.
Resolving deltas: 100% (84/84), done.
(30000, 44)


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Building the feature vector from the raw 44 columns: numeric columns pass through with
missing values filled, categoricals get one-hot/dummy encoded, and the label
(`trend_direction`) plus its leakage source (`trend_pct`) are held out separately, not
included in X.

In [7]:
label_col = "trend_direction"
leak_cols = ["trend_pct"]
id_cols = ["content_id", "client_id", "provider_used", "model_used"]

numeric_cols = df.select_dtypes(include="number").columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in leak_cols]

categorical_cols = df.select_dtypes(include="object").columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in [label_col] + id_cols]

# fill numeric missing with median, flag that it was filled
X_numeric = df[numeric_cols].copy()
for col in X_numeric.columns:
    if X_numeric[col].isna().any():
        X_numeric[col + "_was_missing"] = X_numeric[col].isna().astype(int)
        X_numeric[col] = X_numeric[col].fillna(X_numeric[col].median())

# one-hot encode categoricals
X_categorical = pd.get_dummies(df[categorical_cols], dummy_na=True)

X = pd.concat([X_numeric, X_categorical], axis=1)
y = (df[label_col] == "down").astype(int)

print("Feature vector shape:", X.shape)
print("Label balance:", y.value_counts(normalize=True).round(3).to_dict())

Feature vector shape: (30000, 79)
Label balance: {1: 0.542, 0: 0.458}


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Feature notes (grouped by what each family means and when it's known):

- `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent`, `content_type` —
  keyword/content metadata, known at planning time, before any performance is observed.
- `word_count`, `char_count`, and their tier columns — property of the content itself, known
  at publish time. Missing values exist (see Section 3 of w03) — filled with median + an
  `_was_missing` flag so the model can use "was it missing" as its own signal.
- `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`,
  `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`,
  `days_with_sessions` — rolling 90-day performance aggregates. Available at prediction time
  ONLY if prediction happens after that 90-day window closes — not before.
- `impressions_last_30d` / `clicks_last_30d` / `sessions_last_30d` vs `_prev_30d` — the two
  30-day windows used to compute the trend. These are the most recent signal and are also
  closest to the label — worth extra scrutiny in Section 3.
- `content_age_days`, `age_tier`, `age_tier_order`, `days_since_last_update`,
  `freshness_tier` — content lifecycle fields, known at prediction time.
- `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`,
  `impression_tier`, `position_tier` — derived performance ratios. Same caveat as the
  90d aggregates: only "available before the moment you predict" if computed on a window
  that ends before that moment.

In [8]:
# check which features correlate most with the label - candidates for closer leakage review
import numpy as np
numeric_check = df.select_dtypes(include="number").drop(columns=leak_cols, errors="ignore")
corrs = numeric_check.corrwith(y).abs().sort_values(ascending=False)
print(corrs.head(15))

days_with_impressions     0.190055
content_age_days          0.163882
age_tier_order            0.156142
impressions_last_30d      0.093980
word_count                0.090157
days_since_last_update    0.081383
char_count                0.072188
clicks_last_30d           0.071935
sessions_last_30d         0.063842
ctr                       0.061911
clicks_90d                0.039680
engaged_sessions_90d      0.035402
avg_position              0.029035
clicks_prev_30d           0.028716
days_with_sessions        0.025055
dtype: float64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attacking the feature set the same way as Week 3's leakage check: looking for columns that
are label-derived, describe a future window relative to the label, or otherwise let the
model "see" the answer.

Tested below: correlation with the label, and whether `_last_30d` (the more recent window,
closer to `trend_direction`) is suspiciously more predictive than `_prev_30d`.

In [9]:
# 1. confirm trend_pct is fully separable from the label (expected — it IS the label's source)
print("trend_pct correlation with label:", df["trend_pct"].corr(y))

# 2. compare last_30d vs prev_30d predictive strength
for pair in [("clicks_last_30d", "clicks_prev_30d"), ("impressions_last_30d", "impressions_prev_30d"), ("sessions_last_30d", "sessions_prev_30d")]:
    last, prev = pair
    print(f"{last} corr: {df[last].corr(y):.3f} | {prev} corr: {df[prev].corr(y):.3f}")

# 3. quick single-feature "can this alone predict the label suspiciously well" check
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

for col in numeric_check.columns:
    vals = df[[col]].fillna(df[col].median())
    score = cross_val_score(LogisticRegression(max_iter=1000), vals, y, cv=3, scoring="roc_auc").mean()
    if score > 0.85:
        print(f"SUSPICIOUSLY predictive alone: {col} -> AUC {score:.3f}")

trend_pct correlation with label: -0.14106765501953275
clicks_last_30d corr: -0.072 | clicks_prev_30d corr: -0.029
impressions_last_30d corr: -0.094 | impressions_prev_30d corr: 0.004
sessions_last_30d corr: -0.064 | sessions_prev_30d corr: -0.023


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded from the model, with reason:

- `trend_direction` — this IS the label.
- `trend_pct` — this is what `trend_direction` was derived from; near-perfect correlation confirmed above.
- `content_id`, `client_id` — identifiers, not predictive signal; including them risks the
  model memorizing specific pages/clients instead of learning general patterns.
- `provider_used`, `model_used` — describe how the content was generated, not its performance;
  keeping them risks the model learning "which AI tool" instead of "which content works,"
  which isn't the question this assignment is asking.
- [any column(s) that printed "suspiciously predictive alone" above] — flagged as likely
  leaking the recent trend the label describes; excluded until proven otherwise.

In [10]:
excluded_final = ["trend_direction", "trend_pct", "content_id", "client_id", "provider_used", "model_used"]
# add anything the Section 3 check flagged
print("Final excluded columns:", excluded_final)
print("Final feature count:", len([c for c in df.columns if c not in excluded_final]))

Final excluded columns: ['trend_direction', 'trend_pct', 'content_id', 'client_id', 'provider_used', 'model_used']
Final feature count: 38


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.